# Classify TSOSI supporters

- goal : add a new level of semantic to describe TSOSI supporters
- goal : find a way to identify _library consortium_, _research funding org_, _resaerch perf. org_, and _publisher_
- ROR types metadata. [see the doc](https://ror.readme.io/docs/ror-data-structure#types) they are not exclusive : `education, funder, healthcare, company, archive, nonprofit, government, facility, and other`

## 1. library consortium

**definition of lib consort**
- an organisaiton that regroup research libraries and offer scholarly communication services to its member.
- Q: What if the org. is more a library scope, eg. Big 10? grouping institution

**get data**
- one source is the "international coalition of lib consortia". Here is a list of their members and most of them are library consortia:  https://icolc.net/participating-consortia?page=3. but not all are lib consoirtia
- starting from TSOSI data and we can list all our "intermediary", but not all are a library consort like CCSD/JISC for example
- payment level : something that faciliate the link between infra & supporter
- academic level : an organization with members taht are mostly reserach libraries ; one goal of the organisation is to support OS infra.
- specificy : the org can be led by one research institution (eg. FinElib)


In [2]:
import pandas as pd, json, requests

In [3]:
raw_entities = requests.get("https://tsosi.org/api/entities/all?format=json").json()
len(raw_entities)

1178

In [4]:
## flatten all entities
for item in raw_entities : 
    if item.get("identifiers") :
        for ids in item["identifiers"] : 
            if ids["registry"] == "ror" : 
                item["ror_id"] = ids["value"]
            if ids["registry"] == "wikidata" : 
                item["wiki_id"] = ids["value"]

In [5]:
df_raw  = pd.json_normalize(raw_entities)

In [6]:
df_raw.columns

Index(['id', 'name', 'short_name', 'country', 'identifiers', 'coordinates',
       'logo', 'icon', 'is_emitter', 'is_agent', 'is_recipient', 'is_partner',
       'is_scoss', 'is_posi', 'is_barcelona', 'wiki_id', 'ror_id'],
      dtype='object')

In [9]:
print(json.dumps(df_raw.iloc[10].to_dict(), ensure_ascii=False, indent=2))

{
  "id": "79ca5f77-4533-41e4-a807-02887e9d0dc6",
  "name": "Érudit",
  "short_name": "Érudit",
  "country": "CA",
  "identifiers": [
    {
      "registry": "tsosi",
      "value": "T681249"
    },
    {
      "registry": "ror",
      "value": "05xay3m79"
    },
    {
      "registry": "wikidata",
      "value": "Q3591464"
    }
  ],
  "coordinates": "POINT(-73.58781 45.50884)",
  "logo": "https://tsosi.org/media/79ca5f77-4533-41e4-a807-02887e9d0dc6/logo/Logo_of_Erudit.svg",
  "icon": "https://tsosi.org/media/79ca5f77-4533-41e4-a807-02887e9d0dc6/icon/erudit_icon_PL3JWeF.png",
  "is_emitter": true,
  "is_agent": false,
  "is_recipient": true,
  "is_partner": false,
  "is_scoss": false,
  "is_posi": false,
  "is_barcelona": true,
  "wiki_id": "Q3591464",
  "ror_id": "05xay3m79"
}


In [58]:
df_consort = df_raw[df_raw["is_agent"] == True ][["name", "country","ror_id", "wiki_id", "is_barcelona", "id"]].reset_index(drop=True)
df_consort.sample(3)

,name,country,ror_id,wiki_id,is_barcelona,id
42,Sachsen Konsortium,DE,NaN,Q134580046,False,a94f5281-7f00-4f8a-af45-b0e169c239e2
8,Canadian Research Knowledge Network,CA,00s5kmw74,Q76609347,False,6c18e29a-6650-401b-81de-ef8ffee5c897
23,HeBIS-Konsortium,DE,NaN,Q76612424,False,cb296def-6aaf-4070-ae3b-2868caef94c6


In [7]:
def get_ror_types(ror_id) :
    "return the ror types of the entity"
    ror_data = requests.get(f"https://api.ror.org/v2/organizations/{ror_id}").json()
    if ror_data.get("types") : 
        return ", ".join(ror_data.get("types"))

In [72]:
## only to test the function
get_ror_types("00te3t702")

'education, funder'

In [73]:
df_consort["ror_types"] = df_consort.apply(
    lambda x: get_ror_types(x["ror_id"]) if pd.notna(x["ror_id"]) else "",
    axis=1
)

In [1]:
df_consort.to_csv("TSOSI-identify-lib-consort.csv", index = False)

NameError: name 'df_consort' is not defined

### Find agent via the transfers endpoint

In [40]:
raw_transfers = requests.get("https://tsosi.org/api/transfers/all?format=json").json()
len(raw_transfers)

4783

In [41]:
df_transfers = pd.json_normalize(raw_transfers)
## reduce to transfers with agents
mask = df_transfers['agent_ids'].apply(lambda x: isinstance(x, list) and len(x) > 0)
## apply filter and keep only wanted cols
df_intermed = df_transfers[mask].reset_index(drop=True)[["emitter_id", "agent_ids", "date_clc.value", "amounts_clc.EUR"]]
df_intermed.shape

(2960, 4)

In [43]:
## agent may contains many item, repeat row for each item when it has many occur
df_intermed_list = df_intermed.explode('agent_ids').reset_index(drop=True)
df_intermed_list.shape

(2963, 4)

In [47]:
## regroupe by agent
df_intermed_meta = df_intermed_list.groupby('agent_ids').agg(
        nb_emitters=('emitter_id', 'nunique'),
        nb_transfers=('date_clc.value', 'count'),
).reset_index()
df_intermed_meta.columns

Index(['agent_ids', 'nb_emitters', 'nb_transfers'], dtype='object')

In [48]:
df_intermed_meta.shape

(50, 3)

In [13]:
print(json.dumps(raw_entities[10], ensure_ascii=False, indent=2))

{
  "id": "79ca5f77-4533-41e4-a807-02887e9d0dc6",
  "name": "Érudit",
  "short_name": "Érudit",
  "country": "CA",
  "identifiers": [
    {
      "registry": "tsosi",
      "value": "T681249"
    },
    {
      "registry": "ror",
      "value": "05xay3m79"
    },
    {
      "registry": "wikidata",
      "value": "Q3591464"
    }
  ],
  "coordinates": "POINT(-73.58781 45.50884)",
  "logo": "https://tsosi.org/media/79ca5f77-4533-41e4-a807-02887e9d0dc6/logo/Logo_of_Erudit.svg",
  "icon": "https://tsosi.org/media/79ca5f77-4533-41e4-a807-02887e9d0dc6/icon/erudit_icon_KB0bxb7.png",
  "is_emitter": true,
  "is_agent": false,
  "is_recipient": true,
  "is_partner": false,
  "is_scoss": false,
  "is_posi": false,
  "is_barcelona": true
}


## 2. find research funder

- start by collecting org name as sample on ScienceEurope (national research funding org) cf. https://www.scienceeurope.org/about-us/members/?type=Research%20Funding%20Organisation%20(RFO)
- 

In [3]:
import pandas as pd, json, requests

In [4]:
raw_entities = requests.get("https://tsosi.org/api/entities/all?format=json").json()
len(raw_entities)

1124

In [5]:
## flatten all entities
for item in raw_entities : 
    if item.get("identifiers") :
        for ids in item["identifiers"] : 
            if ids["registry"] == "ror" : 
                item["ror_id"] = ids["value"]
            if ids["registry"] == "wikidata" : 
                item["wiki_id"] = ids["value"]

In [6]:
df_raw  = pd.json_normalize(raw_entities)

In [ ]:
df_raw["ror_types"] = df_raw.apply(
    lambda x: get_ror_types(x["ror_id"]) if pd.notna(x["ror_id"]) else "",
    axis=1
)

In [30]:
df_raw.columns

Index(['id', 'name', 'short_name', 'country', 'identifiers', 'coordinates',
       'logo', 'icon', 'is_emitter', 'is_agent', 'is_recipient', 'is_partner',
       'is_scoss', 'is_posi', 'is_barcelona', 'wiki_id', 'ror_id',
       'ror_types'],
      dtype='object')

In [39]:
df_funders = df_raw[["name", "short_name", "country", "ror_id", "ror_types"]].copy()

In [49]:
## explore with funder & government, funder & nonprofit, funder&company
df_funders[ 
     df_funders["ror_types"].str.contains(r"\bfunder\b", case=False, na=False) & 
     df_funders["ror_types"].str.contains(r"\bfacility\b", case=False, na=False) 
     ]

,name,short_name,country,ror_id,ror_types
160,Centre Méditerranéen de l’Environnement et de ...,None,FR,04eeyfe07,"facility, funder"
219,Département sciences pour l'ingénierie des ali...,None,FR,00sxj6k90,"facility, funder"
264,European Organization for Nuclear Research,None,CH,01ggx4157,"facility, funder"
283,Forschungszentrum Jülich,None,DE,02nv7yv05,"facility, funder"
303,GFZ Helmholtz Centre for Geosciences,None,DE,04z8jg394,"facility, funder"
347,Ifremer,None,FR,044jxhp58,"facility, funder"
357,Institut des Sciences de l'Evolution de Montpe...,None,FR,01cah1n37,"facility, funder"
359,Institute for Advanced Study,None,US,00f809463,"facility, funder"
364,Institute of Physics,None,GB,04whb3988,"facility, funder"
410,Korea Institute of Science & Technology Inform...,None,KR,01k4yrm29,"facility, funder"


In [ ]:
df_funders[ df_funders["ror_types"].str.contains(r"\barchive\b", case=False, na=False) ]

In [38]:
df_funders.shape

(617, 5)

## 3. what about company category

forget the type of ROR it's not accuratye. eg. PLOS is not in "company" but in "archive" (only) type
PLOS is a company, but it's a not for profit one, like ACS, lilke OCLC (not for profit and funder record in ROR)